In [1]:
import os
os.getcwd()

'c:\\Users\\param\\projects\\text-summarizer\\research'

In [2]:
os.chdir("../")
os.getcwd()

'c:\\Users\\param\\projects\\text-summarizer'

In [3]:
import os
import sys
import logging

import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import create_optimizer
from datasets import load_from_disk

from textSummarizer import logger
from textSummarizer.exception import CustomException
from textSummarizer.entity import ModelTrainerConfig
from textSummarizer.config.configuration import ConfigurationManager

c:\Users\param\projects\text-summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
config_manager = ConfigurationManager()
model_trainer_config = config_manager.get_data_model_trainer_config()
print(model_trainer_config)

[2026-07-04 03:29:14,738: INFO: 26: common: yaml file (config\config.yaml) is loaded successfully]
[2026-07-04 03:29:14,738: INFO: 26: common: yaml file (params.yaml) is loaded successfully]
[2026-07-04 03:29:14,738: INFO: 43: common: Created directory at (artifacts)]
[2026-07-04 03:29:14,738: INFO: 43: common: Created directory at (artifacts/model_trainer)]
ModelTrainerConfig(root_dir='artifacts/model_trainer', data_path='artifacts/data_transformation/samsum_dataset', model_ckpt='google/pegasus-cnn_dailymail', num_train_epoch=1, warmup_steps=500, batch_size=2, learning_rate='2e-5', weight_decay=0.01)


In [7]:
class ModelTrainer:
    def __init__(self, config : ModelTrainerConfig) -> ModelTrainerConfig:
        self.config = config
    
    def train(self):
        try:
            
            tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
            model = TFAutoModelForSeq2SeqLM.from_pretrained(
                self.config.model_ckpt,
                from_pt = True
            )

            datacollator = DataCollatorForSeq2Seq(
                tokenizer,
                model=model,
                return_tensors='tf',
                padding = True
            )

            dataset = load_from_disk(self.config.data_path)

            train_dataset = dataset['train'].to_tf_dataset(
                columns=['input_ids','attention_mask','labels'],
                shuffle=True,
                batch_size=self.config.batch_size,
                collate_fn=datacollator
            )
            eval_dataset = dataset['validation'].to_tf_dataset(
                columns= ['input_ids', 'attention_mask', 'label'],
                shuffle=False,
                batch_size=self.config.batch_size,
                collate_fn=datacollator
            )

            num_train_steps = len(train_dataset)*int(self.config.num_train_epoch)

            optimizer,_ = create_optimizer(
            init_lr=float(self.config.learning_rate),
            num_warmup_steps=int(self.config.warmup_steps),
            num_train_steps=int(num_train_steps),
            weight_decay_rate=float(self.config.weight_decay)                
            )

            model.compile(optimizer=optimizer)
            model.fit(
                train_dataset,
                validation_data=eval_dataset,
                epochs=self.config.num_train_epoch
            )
            model.save_pretained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
            tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))
            logging.info("Model training completed and saved successfully")

        except Exception as e:
            raise CustomException(e,sys)

In [ ]:
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

All PyTorch model weights were used when initializing TFPegasusForConditionalGeneration.

Some weights or buffers of the TF 2.0 model TFPegasusForConditionalGeneration were not initialized from the PyTorch model and are newly initialized: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[2026-07-04 03:32:11,343: WARNING: 149: module_wrapper: From c:\Users\param\projects\text-summarizer\.venv\Lib\site-packages\tf_keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.
]
   4/7366 [..............................] - ETA: 69:52:36 - loss: 3.3590